# Sperimentazione con Architetture Encoder-Only (Famiglia BERT)

Questo notebook è stato configurato per esplorare le differenze pratiche e architetturali tra diverse varianti di modelli BERT in task di classificazione testuale.
Il codice è strutturato per sfruttare l'accelerazione hardware (CUDA) per massimizzare le performance della tua RTX 5070.

### Librerie necessarie
Assicurati di avere installato: `pip install torch transformers datasets tqdm scikit-learn`

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score

# Configurazione Device
print(torch.__version__, torch.version.cuda)
print(torch.cuda.get_arch_list())   # deve contenere 'sm_120'
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo in uso: {device}")
if torch.cuda.is_available():
    print(f"GPU rilevata: {torch.cuda.get_device_name(0)}")
    print(f"Memoria VRAM totale: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

2.11.0+cu128 12.8
['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
True NVIDIA GeForce RTX 5070
Dispositivo in uso: cuda
GPU rilevata: NVIDIA GeForce RTX 5070
Memoria VRAM totale: 11.94 GB


In [7]:
def evaluate_model(model_name, dataset_name, text_column, label_column, num_labels=2, batch_size=32, max_samples=1000, dataset_config_name=None):
    """
    Funzione helper per scaricare un modello, tokenizzare un subset di dati 
    ed estrarre le predizioni per valutarne l'accuratezza zero-shot / pre-finetuned.
    """
    print(f"\n--- Caricamento di {model_name} ---")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
    model.to(device)
    model.eval()
    
    print(f"Caricamento dataset: {dataset_name} " + (f"(config: {dataset_config_name})" if dataset_config_name else ""))
    
    # Risoluzione robusta dello split (test -> validation -> validation_matched)
    try:
        dataset = load_dataset(dataset_name, dataset_config_name, split=f'test[:{max_samples}]')
    except ValueError:
        try:
            dataset = load_dataset(dataset_name, dataset_config_name, split=f'validation[:{max_samples}]')
        except ValueError:
            dataset = load_dataset(dataset_name, dataset_config_name, split=f'validation_matched[:{max_samples}]')
    
    def tokenize_function(examples):
        return tokenizer(examples[text_column], padding='max_length', truncation=True, max_length=128)
    
    tokenized_datasets = dataset.map(tokenize_function, batched=True)
    tokenized_datasets.set_format('torch', columns=['input_ids', 'attention_mask', label_column])
    
    dataloader = DataLoader(tokenized_datasets, batch_size=batch_size)
    
    true_labels = []
    predictions = []
    
    print("Esecuzione inferenza...")
    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch[label_column].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1)
            
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
            
    acc = accuracy_score(true_labels, predictions)
    print(f"Accuratezza sul subset: {acc:.4f}")
    return acc


## 1. BERT (Bidirectional Encoder Representations from Transformers)

**Modello:** `textattack/bert-base-uncased-imdb` (Fine-tuned per sentiment analysis)
**Dataset:** IMDB (Recensioni di film - classificazione binaria Positivo/Negativo)

**Caratteristiche Teoriche:**
BERT è il progenitore della famiglia. È stato addestrato su due task principali:
1. **Masked Language Modeling (MLM):** Maschera casualmente il 15% dei token e cerca di predirli. Permette di catturare il contesto bidirezionale.
2. **Next Sentence Prediction (NSP):** Cerca di capire se la frase B segue logicamente la frase A.

Ha un'architettura molto solida (110M parametri nella versione base) ed è storicamente il modello più studiato nell'Explainable AI, il che significa che l'estrazione delle rappresentazioni latenti e l'interpretazione dei layer di attenzione sono ampiamente documentate.

In [8]:
# Testiamo BERT su un task di Sentiment Analysis (IMDB)
evaluate_model(
    model_name='textattack/bert-base-uncased-imdb',
    dataset_name='stanfordnlp/imdb',
    text_column='text',
    label_column='label',
    num_labels=2,
    batch_size=64,  # La 5090 può gestire batch ben più grossi, ma teniamo un valore conservativo per la memoria di base
    max_samples=1000
)


--- Caricamento di textattack/bert-base-uncased-imdb ---


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 47522.84it/s]


Caricamento dataset: stanfordnlp/imdb 


c:\Users\ricca\Desktop\Università\Magistrale\3° semestre\Explainable AI - Progetto\Notebooks\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ricca\.cache\huggingface\hub\datasets--stanfordnlp--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 1000/1000 [00:00<00:00, 8149.9

Esecuzione inferenza...


100%|██████████| 16/16 [00:01<00:00,  8.67it/s]

Accuratezza sul subset: 0.8750


0.875

## 2. DistilBERT

**Modello:** `bhadresh-savani/distilbert-base-uncased-emotion` (Fine-tuned per classificazione emozioni)
**Dataset:** Dair-ai/emotion (Classificazione multi-classe: gioia, tristezza, rabbia, ecc.)

**Caratteristiche Teoriche:**
DistilBERT utilizza la tecnica della **Knowledge Distillation** (distillazione della conoscenza). 
Durante il pre-training, apprende non solo dalle etichette, ma cerca di imitare la distribuzione probabilistica generata da un modello "insegnante" più grande (BERT-base).

Il risultato è un modello che:
- È del 40% più leggero (circa 66M di parametri).
- È del 60% più veloce in inferenza.
- Mantiene il 97% delle performance di language understanding di BERT.

Nel contesto dell'XAI (come il progetto P9), DistilBERT è un'ottima scelta per iterare velocemente, poiché i suoi 6 layer (invece di 12) dimezzano i tempi per il calcolo delle derivate direzionali (TCAV).

In [9]:
# Testiamo DistilBERT su un task di classificazione di emozioni
evaluate_model(
    model_name='bhadresh-savani/distilbert-base-uncased-emotion',
    dataset_name='dair-ai/emotion',
    text_column='text',
    label_column='label',
    num_labels=6, # sadness, joy, love, anger, fear, surprise
    batch_size=64,
    max_samples=1000
)


--- Caricamento di bhadresh-savani/distilbert-base-uncased-emotion ---


c:\Users\ricca\Desktop\Università\Magistrale\3° semestre\Explainable AI - Progetto\Notebooks\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ricca\.cache\huggingface\hub\models--bhadresh-savani--distilbert-base-uncased-emotion. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|█

Caricamento dataset: dair-ai/emotion 


c:\Users\ricca\Desktop\Università\Magistrale\3° semestre\Explainable AI - Progetto\Notebooks\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ricca\.cache\huggingface\hub\datasets--dair-ai--emotion. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 1000/1000 [00:00<00:00, 18993.9

Esecuzione inferenza...


100%|██████████| 16/16 [00:01<00:00, 15.62it/s]

Accuratezza sul subset: 0.9220


0.922

## 3. RoBERTa (Robustly Optimized BERT Approach)

**Modello:** `roberta-large-mnli` (Fine-tuned per Natural Language Inference)
**Dataset:** GLUE (MNLI) - (Classificazione logica: entailment, neutral, contradiction)

**Caratteristiche Teoriche:**
RoBERTa è nato come un'ottimizzazione rigorosa di BERT da parte di Facebook AI (Meta). 
Le differenze chiave includono:
1. **Rimozione del task NSP:** Si sono accorti che la Next Sentence Prediction ostacolava le performance.
2. **Dynamic Masking:** Invece di mascherare i token una sola volta durante la preparazione dei dati, genera maschere nuove a ogni epoca di addestramento.
3. **Training potenziato:** È stato addestrato su molti più dati (160GB vs 16GB di BERT) e con batch size enormi.

RoBERTa-large (circa 355M di parametri) è più pesante, ma spesso batte BERT in tutti i task a parità di contesto. Sulla tua GPU girerà senza alcun problema.

In [11]:
# Testiamo RoBERTa su un task NLI (Natural Language Inference)
evaluate_model(
    model_name='roberta-large-mnli',
    dataset_name='nyu-mll/glue',       # <-- Aggiornato con il namespace ufficiale
    dataset_config_name='mnli',
    text_column='premise', # Nota: per l'inferenza base su NLI servirebbero premise+hypothesis, usiamo una semplificazione per il test del codice
    label_column='label',
    num_labels=3,
    batch_size=32,
    max_samples=1000
)


--- Caricamento di roberta-large-mnli ---


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 741.43it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Caricamento dataset: nyu-mll/glue (config: mnli)


c:\Users\ricca\Desktop\Università\Magistrale\3° semestre\Explainable AI - Progetto\Notebooks\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ricca\.cache\huggingface\hub\datasets--nyu-mll--glue. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 1000/1000 [00:00<00:00, 22888.93 e

Esecuzione inferenza...


100%|██████████| 32/32 [00:05<00:00,  5.74it/s]

Accuratezza sul subset: 0.3320


0.332